# Quipu Extraction (NB 60)

One-shot scan + extraction pipeline. Builds:

- `data/quipu_data.csv` — one row per inscription (metadata, dimensions, body_file pointer, canonical_status)
- `data/bodies/{root_txid}.bin` — raw concat'd header+body bytes per inscription
- `data/tx_inputs.csv` — slim (txid, inputs) table for the wallet's transactions, used by the funding-edges step
- `data/quipu_edges.csv` — funding / keydrop / citation edges between quipus

Re-run end-to-end to refresh after new inscriptions land on chain. The **last cell is self-contained** — given the data/ files exist, it can be run on a fresh kernel without re-running any earlier cells.

## Setup — RPC config + the 9 watched addresses

In [ ]:
import os, sys, json, time
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import pandas as pd
from colegio_tools import rpc_request

DATA_DIR    = os.path.join(REPO, 'data')
BODIES_DIR  = os.path.join(DATA_DIR, 'bodies')
os.makedirs(BODIES_DIR, exist_ok=True)

ADDRESSES = {
    '9xth7DcLGb1nACScMBeSfDCfghhLKF7yqs': 'bordado',
    'D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX': 'apocrypha',
    'A7pfCe2Cw9JD2C4vEZbpDmUZJy7B2TaefV': 'ha',
    'AD28bxzxyrd3a4Qgad2VNQ2eN5Leg8ozuw': 'ca',
    'A3ShjwjsAE4ysM66EZJM3A28tPnL2jNDgC': 'multiman',
    'A3ABo52FjMJ57KSjbKyfe9aiKkH2jntXHY': 'test_multisig3',
    'DPy94XwsHvFpXfA2C6PjERknyNjQYacufZ': 'test1',
    'D6hcCyELYoMgPiMUfjGgBSHNSdZWrULupx': 'test2',
    'DPJAJuNW9ajjnEUy9RhYDjoMB9aFmkLdDb': 'test3',
}
ADDRESS_LIST = list(ADDRESSES.keys())
print(f'tip: {rpc_request("getblockcount")}')
for addr, label in ADDRESSES.items():
    print(f'  {label:16s} {addr}')

## Step 1 — Scan: build `df_transactions` + `df_outputs`

Pull all wallet txs once, then for each unique txid fetch the raw transaction and populate the outputs frame with `spent_in` cross-references.

In [ ]:
from colegio_tools import extract_op_return

print('pulling wallet events...')
all_events = rpc_request('listtransactions', ['*', 200000, 0, True])
print(f'  {len(all_events)} events')

txids_per_addr = {a: set() for a in ADDRESS_LIST}
for e in all_events:
    a = e.get('address')
    if a in txids_per_addr:
        txids_per_addr[a].add(e['txid'])

all_txids = set()
for s in txids_per_addr.values():
    all_txids |= s
print(f'  {len(all_txids)} unique txids across all watched addresses')

In [ ]:
# Fetch raw tx detail for each unique txid (expensive — runs once)
print(f'fetching {len(all_txids)} raw txs...')
detailed = []
blockheight_cache = {}
for i, txid in enumerate(all_txids):
    try:
        raw = rpc_request('getrawtransaction', [txid, 1])
    except Exception as e:
        print(f'  skip {txid[:12]}…: {e}')
        continue
    bh = raw.get('blockhash')
    if bh and bh in blockheight_cache:
        height = blockheight_cache[bh]
    elif bh:
        height = rpc_request('getblock', [bh])['height']
        blockheight_cache[bh] = height
    else:
        height = None
    op_ret = None
    for v in raw.get('vout', []):
        d = extract_op_return(v)
        if d:
            op_ret = d
            break
    detailed.append({
        'txid':         txid,
        'blockhash':    bh,
        'blockheight':  height,
        'blocktime':    raw.get('blocktime'),
        'inputs':       [f"{vin['txid']}:{vin['vout']}" for vin in raw.get('vin', []) if 'txid' in vin],
        'values':       [v['value'] for v in raw.get('vout', [])],
        'num_inputs':   len(raw.get('vin', [])),
        'num_outputs':  len(raw.get('vout', [])),
        'op_return':    op_ret,
    })
    if (i+1) % 5000 == 0:
        print(f'  {i+1} / {len(all_txids)}')

df_transactions = pd.DataFrame(detailed).sort_values(['blockheight','blocktime']).reset_index(drop=True)
print(f'df_transactions: {len(df_transactions)} rows')

In [ ]:
# Build df_outputs: one row per (txid, vout) with spent_in cross-reference.
# Convention: every output row of a tx carries the same op_return value
# (matches what identify_quipus expects from quipu3.ipynb).
rows = []
for _, tx in df_transactions.iterrows():
    for n in range(tx['num_outputs']):
        rows.append({
            'txout':       f"{tx['txid']}:{n}",
            'spent_in':    None,
            'value':       tx['values'][n] if n < len(tx['values']) else None,
            'op_return':   tx['op_return'],
            'blockheight': tx['blockheight'],
            'blocktime':   tx['blocktime'],
            'txid':        tx['txid'],
            'n':           n,
        })
df_outputs = pd.DataFrame(rows)

txout_to_idx = {row['txout']: idx for idx, row in df_outputs.iterrows()}
for _, tx in df_transactions.iterrows():
    for inp in tx['inputs']:
        idx = txout_to_idx.get(inp)
        if idx is not None:
            df_outputs.at[idx, 'spent_in'] = tx['txid']

df_outputs = df_outputs.sort_values(['blockheight','blocktime']).reset_index(drop=True)
df_outputs['op_return'] = df_outputs['op_return'].fillna('')
df_outputs['spent_in']  = df_outputs['spent_in'].fillna('')
print(f'df_outputs: {len(df_outputs)} rows')

## Step 2 — Identify quipu roots

In [ ]:
from colegio_tools import identify_quipus, read_quipu

roots = identify_quipus(df_transactions, df_outputs)
print(f'{len(roots)} quipu root candidates')

## Step 3 — Walk each root and parse the header

For each root, walk the diamond, parse the structural header (per-type), and emit one row per inscription. Bodies go to `data/bodies/{root_txid}.bin`.

In [ ]:
TYPE_NAMES = {
    0x00: 'text', 0x01: 'essay', 0x03: 'image', 0x07: 'audio',
    0x09: 'book',
    0x0e: 'encrypted', 0x1d: 'identity',
    0x3d: 'scene',
    0x5c: 'latex',
    0xab: 'binding', 0xcc: 'cert',
    0xce: 'celestial', 0xee: 'estandarte',
}

# Tone byte names + valid set come from canonical/tone.py (single source
# of truth). A new tone value added there is recognized here automatically
# with no edit to this notebook.
from tone import TONES as TONE_NAMES, VALID_TONES

def parse_dims(blob):
    """Return (dimensions_dict, title, header_length) for any v1 quipu.

    For images, body offset is back-computed from the declared (W, H, color,
    bit_depth) so all historical title conventions parse consistently under
    the lenient title rule.
    """
    if len(blob) < 6 or blob[:4] != b'\xc1\xdd\x00\x01':
        return {}, '', 0
    t = blob[4]
    if t == 0x00:  # text
        hdr_end = 6
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        return {}, title, hdr_end
    if t == 0x01:  # essay (same header grammar as text)
        hdr_end = 6
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        return {}, title, hdr_end
    if t == 0x03:  # image
        if len(blob) < 12: return {}, '', 0
        color = blob[6]; W = (blob[7]<<8)|blob[8]; H = (blob[9]<<8)|blob[10]; bd = blob[11]
        dims = {'color': color, 'W': W, 'H': H, 'bit_depth': bd}
        if color not in (0x00, 0x01):
            return dims, '', 12
        ch = 1 if color == 0 else 3
        expected_body = (W * H * ch * bd + 7) // 8
        body_offset = len(blob) - expected_body
        if body_offset < 12:
            return dims, '', 12
        text = blob[12:body_offset].decode('utf-8', errors='replace')
        if '|' in text:
            parts  = [p.strip() for p in text.split('|')]
            fields = [p for p in parts if p]
            title  = fields[0] if fields else ''
        else:
            cut = text.find('�')
            if cut >= 0:
                text = text[:cut]
            title = text.strip()
        return dims, title, body_offset
    if t == 0x0e:  # encrypted
        if len(blob) < 8: return {}, '', 0
        dims = {'sub_family': blob[6], 'variant': blob[7]}
        hdr_end = 8
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        return dims, title, hdr_end
    if t == 0x5c:  # latex — same pipe-delimited header grammar as text/essay
        hdr_end = 6
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        return {}, title, hdr_end
    if t == 0x09:  # book — same pipe-delimited header grammar as text/essay
        hdr_end = 6
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        # walk past trailing key=value| pairs until the version byte (0x01) begins the body
        while hdr_end < len(blob) and blob[hdr_end] != 0x01:
            nxt = blob.find(b'|', hdr_end)
            if nxt < 0: break
            hdr_end = nxt + 1
        return {}, title, hdr_end
    if t == 0x3d:  # scene — same pipe-delimited header grammar as text/essay
        hdr_end = 6
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        # walk past trailing key=value| pairs until JSON body ({…) begins
        while hdr_end < len(blob) and blob[hdr_end:hdr_end+1] != b'{':
            nxt = blob.find(b'|', hdr_end)
            if nxt < 0: break
            hdr_end = nxt + 1
        return {}, title, hdr_end
    if t == 0xcc:  # cert — title is the first pipe-delimited field in the body
        if len(blob) < 8: return {}, '', 0
        sub = (blob[6]<<8) | blob[7]
        body = blob[8:]
        title = ''
        if body[:1] == b'|':
            close = body.find(b'|', 1)
            if close > 0:
                title = body[1:close].decode('utf-8', errors='replace').strip()
        return {'subtype': sub}, title, 8
    if t == 0xce:  # celestial v1
        if len(blob) < 12: return {}, '', 0
        kind = blob[6]; grouped = blob[7]; meta = blob[8]
        K = (blob[9]<<8) | blob[10]
        T = blob[11]
        title = blob[12:12+T].decode('utf-8', errors='replace') if T else ''
        return {'kind': kind, 'grouped': grouped, 'meta': meta, 'K': K}, title, 12 + T
    if t == 0xee:  # estandarte
        return {}, '', 6
    return {}, '', 0

def find_join_txid(root_txid, df_outputs):
    """Walk every strand to its terminus, find the common tx that spends them all."""
    termini_spenders = []
    n = 0
    while True:
        cur = f'{root_txid}:{n}'
        last_spender = None
        for _ in range(50):
            rows = df_outputs[df_outputs['txout'] == cur]
            if rows.empty: break
            sp = rows.iloc[0]['spent_in']
            if not sp:
                break
            last_spender = sp
            cur = f'{sp}:0'
        if last_spender is None:
            break
        termini_spenders.append(last_spender)
        n += 1
        if n > 32: break
    if not termini_spenders: return None
    from collections import Counter
    c = Counter(termini_spenders)
    most_common, count = c.most_common(1)[0]
    if count >= 2:
        return most_common
    return termini_spenders[-1]

In [ ]:
# Look up which address received each root tx (output 0 typically)
addr_per_root = {}
for root in roots:
    try:
        raw = rpc_request('getrawtransaction', [root, 1])
        addr = raw['vout'][0].get('scriptPubKey', {}).get('addresses', [None])[0]
    except Exception:
        addr = None
    addr_per_root[root] = addr

print(f'looked up address for {sum(1 for v in addr_per_root.values() if v)} of {len(roots)} roots')

In [ ]:
# Walk each root, parse, write body file, accumulate rows.
# Skip identify_quipus heuristic false positives (no v1 magic).
rows = []
skipped_no_magic = []
for root in roots:
    try:
        hh, bh = read_quipu(root, df_outputs=df_outputs)
    except Exception as e:
        rows.append({'root_txid': root, 'notes': f'walk error: {e}'})
        continue
    blob = bytes.fromhex(hh + bh)
    if len(blob) < 6 or blob[:4] != b'\xc1\xdd\x00\x01':
        skipped_no_magic.append((root, blob[:16].hex() if blob else ''))
        continue

    t = blob[4]; tone = blob[5]
    dims, title, hdr_end = parse_dims(blob)
    type_name = TYPE_NAMES.get(t, f'unknown_0x{t:02x}')
    tone_name = TONE_NAMES.get(tone, f'unknown_0x{tone:02x}')

    addr = addr_per_root.get(root)
    label = ADDRESSES.get(addr, '(unknown)')

    tx_row = df_transactions[df_transactions['txid'] == root]
    blockheight = int(tx_row.iloc[0]['blockheight']) if not tx_row.empty and tx_row.iloc[0]['blockheight'] else None
    blocktime   = int(tx_row.iloc[0]['blocktime'])   if not tx_row.empty and tx_row.iloc[0]['blocktime']   else None

    join_txid = find_join_txid(root, df_outputs)

    notes = ''
    if t == 0x03 and dims:
        ch = 1 if dims['color'] == 0 else 3
        expected_body = (dims['W'] * dims['H'] * ch * dims['bit_depth'] + 7) // 8
        actual_body = len(blob) - hdr_end
        if actual_body != expected_body:
            notes = f'image body mismatch: expect {expected_body} B, actual {actual_body} B'

    body_path = os.path.join(BODIES_DIR, f'{root}.bin')
    with open(body_path, 'wb') as f:
        f.write(blob)

    rows.append({
        'root_txid':       root,
        'join_txid':       join_txid or '',
        'address':         addr or '',
        'label':           label,
        'type_byte':       f'0x{t:02x}',
        'type_name':       type_name,
        'tone':            f'0x{tone:02x}',
        'tone_name':       tone_name,
        'title':           title,
        'dimensions_json': json.dumps(dims, sort_keys=True),
        'total_bytes':     len(blob),
        'blockheight':     blockheight,
        'blocktime':       blocktime,
        'body_file':       f'bodies/{root}.bin',
        'notes':           notes,
    })

df_quipus = pd.DataFrame(rows).sort_values(['blockheight','root_txid']).reset_index(drop=True)
print(f'{len(df_quipus)} canonical inscriptions extracted')
print(f'{len(skipped_no_magic)} heuristic false positives skipped (no v1 magic)')
df_quipus.head(20)

## Step 3b — Tag canonical vs pre-canonical compliance

Run strict header-only checks per type and add the `canonical_status` column.

In [ ]:
def check_image_strict(blob):
    # Canonical v1 with the lenient title rule: structural header must be
    # valid and body math must close. Tone validation delegates to the
    # canonical VALID_TONES set imported from tone.py (single source of truth).
    if len(blob) < 12: return False
    if blob[5] not in VALID_TONES: return False
    if blob[6] not in (0x00, 0x01): return False
    bd = blob[11]
    if not (1 <= bd <= 8):          return False
    W = (blob[7]<<8)|blob[8]; H = (blob[9]<<8)|blob[10]
    if W == 0 or H == 0:            return False
    ch = 1 if blob[6] == 0 else 3
    expected = (W * H * ch * bd + 7) // 8
    return len(blob) - expected >= 12

def check_scene_strict(blob):
    # 0x3d scene: header grammar matches text/essay; body must be valid
    # glTF-2.0 JSON with asset.version == "2.0".
    if len(blob) < 7: return False
    if blob[5] not in VALID_TONES: return False
    if blob[6:7] != b'|': return False
    body_start = blob.find(b'|{', 6)
    if body_start < 0: return False
    try:
        import json as _json
        body = _json.loads(blob[body_start+1:].decode('utf-8'))
    except Exception:
        return False
    return isinstance(body, dict) and body.get('asset', {}).get('version') == '2.0'

# All STRICT_CHECKS delegate tone validation to VALID_TONES (imported from
# tone.py in cell 10). A new tone added to canonical/tone.py is recognized
# here automatically.
STRICT_CHECKS = {
    'text':      lambda b: b[5] in VALID_TONES and (len(b)<=6 or b[6:7] != b'|' or b.find(b'|',7) >= 0),
    'essay':     lambda b: b[5] in VALID_TONES and (len(b)<=6 or b[6:7] != b'|' or b.find(b'|',7) >= 0),
    'image':     check_image_strict,
    'cert':      lambda b: len(b)>=8 and b[5] in VALID_TONES and ((b[6]<<8)|b[7]) in (0x0001,0x0002),
    'encrypted': lambda b: len(b)>=8 and b[5] in VALID_TONES and b[6] in (0xae,0xec,0x0d),
    'celestial': lambda b: len(b)>=12 and b[5] in VALID_TONES and b[6] in (0x00,0x01) and b[7] in (0x00,0x01) and b[8] in (0x00,0x01),
    'scene':     check_scene_strict,
}

statuses = []
for _, row in df_quipus.iterrows():
    tname = row['type_name']
    if tname.startswith('unknown_') or tname == 'identity':
        statuses.append('not_yet_canonicalized')
        continue
    if tname not in STRICT_CHECKS:
        statuses.append('not_yet_canonicalized')
        continue
    bpath = os.path.join(DATA_DIR, row['body_file'])
    if not os.path.exists(bpath):
        statuses.append('pre_canonical')
        continue
    blob = open(bpath, 'rb').read()
    statuses.append('canonical_v1' if STRICT_CHECKS[tname](blob) else 'pre_canonical')

df_quipus['canonical_status'] = statuses

from collections import Counter
for k, v in Counter(statuses).most_common():
    print(f'  {k:25s} {v}')

## Step 4 — Save the data

`data/quipu_data.csv` and `data/bodies/*.bin` are the curated outputs. We also save a slim `data/tx_inputs.csv` (just `txid` + `inputs`) so the funding-edges cell at the bottom can run on a fresh kernel without re-scanning.

In [ ]:
csv_path = os.path.join(DATA_DIR, 'quipu_data.csv')
df_quipus.to_csv(csv_path, index=False)
print(f'wrote {csv_path} ({os.path.getsize(csv_path)} bytes)')

body_files = [f for f in os.listdir(BODIES_DIR) if f.endswith('.bin')]
total_body_bytes = sum(os.path.getsize(os.path.join(BODIES_DIR, f)) for f in body_files)
print(f'{len(body_files)} body files, total {total_body_bytes/1024:.1f} KB')

# Slim tx-inputs table — needed by the final cell
slim = df_transactions[['txid', 'inputs']].copy()
slim['inputs'] = slim['inputs'].apply(json.dumps)
slim_path = os.path.join(DATA_DIR, 'tx_inputs.csv')
slim.to_csv(slim_path, index=False)
print(f'wrote {slim_path} ({os.path.getsize(slim_path)/1024:.0f} KB, {len(slim)} txs)')

## Step 5 — Summary by type and address

In [ ]:
summary = df_quipus.groupby(['label', 'type_name']).size().unstack(fill_value=0)
print('--- inscriptions per (label, type) ---')
print(summary)

print('\n--- divergences flagged (excluding heuristic noise) ---')
div = df_quipus[df_quipus['notes'].astype(str).str.len() > 0]
if div.empty:
    print('  none')
else:
    for _, r in div.iterrows():
        print(f'  {r["root_txid"][:12]}…  type={r["type_byte"]}  {r["notes"]}')

## Step 6 — Funding / keydrop / citation edges (self-contained)

**Self-contained**: reads `data/quipu_data.csv` + `data/tx_inputs.csv` + `data/bodies/*.bin` from disk. Runnable on a fresh kernel without running any earlier cells, as long as the data/ files exist (re-run the rest of this notebook end-to-end whenever new inscriptions land on chain).

Edge kinds emitted:
- `funding` — backward walk through input ancestry until hitting another quipu's root or join
- `keydrop` — for `0x0e 0x0d` keydrop inscriptions, an edge from the keydrop to each released target
- `citation_image` / `citation_auth` / `citation` — for text/essay/cert bodies, `<<txid>>` references classified by their field label
- `citation_scene_{content|lock|sound|caption}` — for `0x3d` scene bodies, extracted from glTF `extras.{quipu_ref, lock_ref, sound_ref, caption_quipu_ref}` via `canonical/scene.py`

In [ ]:
import os, sys, re, pandas as pd, json
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

DATA_DIR = os.path.join(REPO, 'data')

df_quipus_local = pd.read_csv(os.path.join(DATA_DIR, 'quipu_data.csv'))
df_quipus_local = df_quipus_local[df_quipus_local['root_txid'].notna()].copy()
all_roots    = set(df_quipus_local['root_txid'])
joins        = set(df_quipus_local['join_txid'].dropna())
join_to_root = {r['join_txid']: r['root_txid'] for _, r in df_quipus_local.iterrows()
                if isinstance(r.get('join_txid'), str) and r['join_txid']}
quipu_struct = all_roots | joins

tx_inputs_path = os.path.join(DATA_DIR, 'tx_inputs.csv')
tx_inputs = {}
if os.path.exists(tx_inputs_path):
    print(f'loading tx_inputs from {tx_inputs_path}')
    df_ti = pd.read_csv(tx_inputs_path)
    for _, r in df_ti.iterrows():
        try:
            raw_inputs = json.loads(r['inputs'])
        except Exception:
            raw_inputs = []
        parsed = [tuple(s.rsplit(':', 1)) for s in raw_inputs]
        tx_inputs[r['txid']] = [(t, int(v)) for t, v in parsed if t]
else:
    raise FileNotFoundError(
        f'{tx_inputs_path} not found — run cells above (steps 1-4) first.'
    )

def trace_back(root_txid, max_hops=15):
    """BFS backward through input ancestry, stopping at any tx not in
    our wallet's tx_inputs (those branches lead outside the quipu graph)."""
    seen = set()
    queue = [(root_txid, 0)]
    hits = []
    while queue:
        txid, hops = queue.pop(0)
        if hops > max_hops or txid in seen:
            continue
        seen.add(txid)
        if hops > 0 and txid in quipu_struct:
            hits.append((txid, hops))
            continue
        ancestors = tx_inputs.get(txid)
        if ancestors is None:
            continue
        for prev_txid, _ in ancestors:
            if prev_txid not in seen:
                queue.append((prev_txid, hops + 1))
    return hits

# === Funding edges ===
funding = []
for _, q in df_quipus_local.iterrows():
    consumer = q['root_txid']
    for anc, hops in trace_back(consumer):
        src_root = anc if anc in all_roots else join_to_root.get(anc)
        if src_root and src_root != consumer:
            funding.append({
                'source_quipu':   src_root,
                'consumer_quipu': consumer,
                'hops':           hops,
                'kind':           'funding',
            })

# === Keydrop edges ===
from encrypted import read_encrypted_quipu
keydrop_edges = []
for _, q in df_quipus_local.iterrows():
    if q['type_name'] != 'encrypted':
        continue
    dims = json.loads(q['dimensions_json'] or '{}')
    if dims.get('sub_family') != 0x0d:
        continue
    blob_path = os.path.join(DATA_DIR, q['body_file'])
    if not os.path.exists(blob_path):
        continue
    blob = open(blob_path, 'rb').read()
    try:
        parsed = read_encrypted_quipu(blob[:8], blob[8:])
    except Exception as e:
        print(f'  keydrop parse failed for {q["root_txid"][:8]}…: {e}')
        continue
    for d in parsed.get('drops', []):
        ref = d.get('ref_txid')
        if ref in all_roots:
            keydrop_edges.append({
                'source_quipu':   q['root_txid'],
                'consumer_quipu': ref,
                'hops':           0,
                'kind':           'keydrop',
            })

# === Citation edges from text + cert bodies ===
CITATION_RE = re.compile(r'<<\s*([0-9a-fA-F]{64})\s*>>')
LABEL_RE    = re.compile(r'(\w+)\s*:\s*<<\s*([0-9a-fA-F]{64})\s*>>')

def body_text_for(q, blob):
    if q['type_name'] in ('text', 'essay'):
        body_offset = 6
        if len(blob) > 6 and blob[6:7] == b'|':
            close = blob.find(b'|', 7)
            if close > 0:
                body_offset = close + 1
        return blob[body_offset:].decode('utf-8', errors='replace')
    if q['type_name'] == 'cert':
        return blob[8:].decode('utf-8', errors='replace')
    return ''

citation_edges = []
seen_cit = set()
for _, q in df_quipus_local.iterrows():
    if q['type_name'] not in ('text', 'essay', 'cert'):
        continue
    blob = open(os.path.join(DATA_DIR, q['body_file']), 'rb').read()
    text = body_text_for(q, blob)
    labeled = {}
    for m in LABEL_RE.finditer(text):
        labeled[m.group(2).lower()] = m.group(1)
    for m in CITATION_RE.finditer(text):
        ref = m.group(1).lower()
        if ref not in all_roots or ref == q['root_txid']:
            continue
        label = labeled.get(ref, '').lower()
        if label == 'image':
            kind = 'citation_image'
        elif label in ('certificateauthority', 'certauthority', 'ca'):
            kind = 'citation_auth'
        else:
            kind = 'citation'
        key = (q['root_txid'], ref, kind)
        if key in seen_cit:
            continue
        seen_cit.add(key)
        citation_edges.append({
            'source_quipu':   q['root_txid'],
            'consumer_quipu': ref,
            'hops':           0,
            'kind':           kind,
        })

# === Scene node-ref edges (0x3d) ===
# Scene bodies use structured glTF extras.{quipu_ref,lock_ref,...}
# rather than <<txid>> citation syntax, so we extract via canonical/scene.py.
from scene import read_scene_quipu, scene_quipu_refs
for _, q in df_quipus_local.iterrows():
    if q['type_name'] != 'scene':
        continue
    blob = open(os.path.join(DATA_DIR, q['body_file']), 'rb').read()
    body_start = blob.find(b'|{', 6)
    if body_start < 0:
        print(f'  scene {q["root_txid"][:8]}…: no JSON body found')
        continue
    try:
        parsed = read_scene_quipu(blob[:body_start+1], blob[body_start+1:])
    except Exception as e:
        print(f'  scene parse failed for {q["root_txid"][:8]}…: {e}')
        continue
    for _node_idx, ref_kind, ref in scene_quipu_refs(parsed):
        ref_lo = ref.lower()
        if ref_lo not in all_roots or ref_lo == q['root_txid']:
            continue
        kind = f'citation_scene_{ref_kind}'
        key = (q['root_txid'], ref_lo, kind)
        if key in seen_cit:
            continue
        seen_cit.add(key)
        citation_edges.append({
            'source_quipu':   q['root_txid'],
            'consumer_quipu': ref_lo,
            'hops':           0,
            'kind':           kind,
        })

all_edges = funding + keydrop_edges + citation_edges
df_edges = pd.DataFrame(all_edges)
df_edges.to_csv(os.path.join(DATA_DIR, 'quipu_edges.csv'), index=False)

from collections import Counter
print(f'\n{len(funding)} funding · {len(keydrop_edges)} keydrop · {len(citation_edges)} citation')
print('citation kinds:', dict(Counter(e["kind"] for e in citation_edges)))

print('\n--- funding ---')
for e in funding:
    s = df_quipus_local[df_quipus_local['root_txid'] == e['source_quipu']].iloc[0]
    d = df_quipus_local[df_quipus_local['root_txid'] == e['consumer_quipu']].iloc[0]
    st = s['title'] if isinstance(s['title'], str) else '(no title)'
    dt = d['title'] if isinstance(d['title'], str) else '(no title)'
    print(f'  {e["source_quipu"][:8]}… "{st[:25]}" --{int(e["hops"])}hop--> {e["consumer_quipu"][:8]}… "{dt[:25]}"')

print('\n--- keydrop ---')
for e in keydrop_edges:
    d = df_quipus_local[df_quipus_local['root_txid'] == e['consumer_quipu']].iloc[0]
    dt = d['title'] if isinstance(d['title'], str) else '(no title)'
    print(f'  {e["source_quipu"][:8]}… (keydrop) ==unlocks==> {e["consumer_quipu"][:8]}… "{dt[:25]}"')

print('\n--- citations ---')
for e in citation_edges:
    s = df_quipus_local[df_quipus_local['root_txid'] == e['source_quipu']].iloc[0]
    d = df_quipus_local[df_quipus_local['root_txid'] == e['consumer_quipu']].iloc[0]
    st = s['title'] if isinstance(s['title'], str) else '(no title)'
    dt = d['title'] if isinstance(d['title'], str) else '(no title)'
    print(f'  [{e["kind"]:22}] {s["type_name"]:9} "{st[:22]}" → {d["type_name"]:9} "{dt[:22]}"')